<div style="display:flex; align-items:center; gap:10px; margin-bottom:8px;">
  <span style="font-size:26px; color:#9558B2;">●</span>
  <span style="font-size:26px; color:#389826;">●</span>
  <span style="font-size:26px; color:#CB3C33;">●</span>
  <span style="font-size:26px; color:#4063D8;">●</span>
  <span style="font-size:30px; font-weight:700; margin-left:6px;">Julia</span>
</div>

# Julia с нуля — **Lesson 8**
## 📘 **Multiple Dispatch** — методы, иерархии типов и диспетчеризация по нескольким аргументам

**Cartesian School · Julia Course**  
**Автор:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School


## Информация об уроке

| Поле | Значение |
|---|---|
| Курс | Julia с нуля |
| Номер урока | Lesson 8 |
| Название | Multiple Dispatch |
| Уровень | Начальный+ / średniozaawansowany |
| Ориентировочное время | 180–240 minut |
| Требования | Lesson 0–7 |
| Темы | funkcje i методы, типy abstrakcyjne i konkretne, single vs multiple dispatch, selekcja najbardziej specyficznej методы, `methods`, `@which`, fallback, ambiguities, parametric methods, `where`, traits, promotion, projektowanie API |
| Автор | Siergej Sobolewski |
| Права | © 2026 Cartesian School |

---

## Plan lekcji

1. Dlaczego multiple dispatch jest ważny.
2. Funkcja a метод.
3. Single dispatch i multiple dispatch.
4. Pierwsze przeciążone методы.
5. Typy abstrakcyjne i konkretne.
6. Najbardziej specyficzna метод.
7. Fallback methods.
8. `methods`.
9. `@which`.
10. `applicable`.
11. `hasmethod`.
12. Dispatch dla wielu аргументów.
13. Пример geometryczny.
14. Parametryczne методы i `where`.
15. Ograniczanie типów.
16. Dispatch na значенияach przez типy.
17. `Val`.
18. Ambiguities.
19. Jak unikać niejednoznaczności.
20. Promotion i dispatch.
21. `convert` i dispatch.
22. Traits.
23. Dispatch a `if x isa`.
24. Dispatch a wydajność.
25. Projektowanie rozszerzalnego API.
26. Typowe błędy.
27. Практика.
28. Мини-проект.
29. Checkpoint.
30. Итоги.


## Учебный стандарт Cartesian School

W całym kursie stosujemy spójny układ:

| Oznaczenie | Znaczenie |
|---|---|
| **Цель** | czego nauczysz się w danym fragmencie |
| **Теория** | definicje i reguły |
| **Пример** | minimalny, działający kod |
| **Анализ** | wyjaśnienie działania |
| **Важно** | zasada, którą trzeba zapamiętać |
| **Типичная ошибка** | częsty błąd i jego przyczyna |
| **Попробуйте сами** | mały eksperyment |
| **Практика** | zadanie do samodzielnego wykonania |
| **Итоги** | najważniejsze wnioski |


## Цели урока

После завершения Lesson 8 вы сможете:

- wyjaśnić różnicę między funkcją a metodą;
- rozumieć, почему multiple dispatch jest centralnym mechanizmem Julia;
- определять wiele metod tej samej функции;
- przewidywać, która метод zostanie wybrana;
- korzystać z `methods`, `@which`, `applicable` i `hasmethod`;
- projektować fallback methods;
- pisać parametryczne методы z `where`;
- rozpoznawać i usuwać ambiguities;
- rozumieć rolę promotion i conversion;
- porównać dispatch z dużym `if x isa`;
- projektować API, które można rozszerzać bez modyfikowania istniejącego kodu.


## **1. Dlaczego multiple dispatch jest ważny?**

### Теория

Julia jest językiem, w którym funkcje nie są „własnością” pojedynczego типu.

Zachowanie programu можетmy определять poprzez **методы**, których wybór zależy od типów wszystkich аргументów функции.

To właśnie nazywamy **multiple dispatch**.


### Intuicja

Załóżmy, że mamy funkcję:

```julia
combine(x, y)
```

Jej zachowanie может zależeć jednocześnie od типu `x` i типu `y`.

Dzięki temu nie musimy budować jednego wielkiego bloku:

```julia
if x isa ...
    if y isa ...
        ...
    end
end
```

Zamiast tego definiujemy osobne методы.


## **2. Funkcja a метод**

### Теория

To podstawowe rozróżnienie:

- **функция** — wspólna nazwa operacji;
- **метод** — конкретная implementacja dla określonego zestawu типów аргументów.


In [ ]:
describe(x::Integer) = "целое число"
describe(x::AbstractFloat) = "число с плавающей точкой"
describe(x::AbstractString) = "строка"

@show describe(10)
@show describe(3.14)
@show describe("Julia")


W tym przykładzie istnieje jedna функция `describe`, ale kilka metod.


## **3. Single dispatch i multiple dispatch**

### Single dispatch

W wielu językach obiektowych метод jest выбираетna głównie na podstawie типu jednego obiektu — zwykle tego po lewej stronie kropki.


### Multiple dispatch

W Julia wybór может zależeć od **wszystkich аргументów**.


In [ ]:
interact(x::Integer, y::Integer) = "Integer + Integer"
interact(x::Integer, y::AbstractString) = "Integer + String"
interact(x::AbstractString, y::Integer) = "String + Integer"
interact(x::AbstractString, y::AbstractString) = "String + String"

@show interact(1, 2)
@show interact(1, "Julia")
@show interact("Julia", 1)
@show interact("Julia", "Lang")


## **4. Pierwsze przeciążone методы**

### Пример


In [ ]:
combine(x::Integer, y::Integer) = x + y
combine(x::AbstractString, y::AbstractString) = x * y

@show combine(10, 20)
@show combine("Julia ", "Language")


### Анализ

Ta sama nazwa `combine` reprezentuje różne znaczenia w зависимости od типów аргументów.


## **5. Typy abstrakcyjne i konkretne**

### Теория

Julia posiada hierarchię типów.

Примерowo:

```text
Number
├── Real
│   ├── Integer
│   └── AbstractFloat
└── Complex
```

Typy abstrakcyjne opisują **rodzinę типów**, a типy konkretne reprezentują faktyczne типy значения.


In [ ]:
@show Int <: Integer
@show Integer <: Real
@show Float64 <: AbstractFloat
@show Float64 <: Real


### Важно

Metoda przyjmująca `Real` может działać dla wielu типów чиселowych, np. `Int`, `Float64`, `BigFloat`.

Nie trzeba определять osobnej методы dla każdego типu konkretnego, jeśli zachowanie jest takie samo.


## **6. Najbardziej specyficzna метод**

### Теория

Jeżeli pasuje kilka metod, Julia выбирает metodę **najbardziej specyficzną**.


In [ ]:
kind(x::Number) = "Number"
kind(x::Real) = "Real"
kind(x::Integer) = "Integer"
kind(x::Int) = "Int"

@show kind(1)
@show kind(Int8(1))
@show kind(1.0)
@show kind(1 + 2im)


### Анализ

Dla `1::Int` pasują wszystkie cztery методы, ale `Int` jest najbardziej specyficzne.


## **7. Fallback methods**

### Теория

Fallback method to метод общая używana wtedy, gdy nie istnieje bardziej specyficzna wersja.


In [ ]:
describe_типe(x::Integer) = "integer"
describe_типe(x::AbstractString) = "string"
describe_типe(x) = "other"

@show describe_типe(10)
@show describe_типe("Julia")
@show describe_типe([1, 2, 3])


### Хорошая практика

Fallback следует mieć sens semantyczny.

Nie dodawaj `f(x) = ...` tylko po to, aby ukryć błąd projektu типów.


## **8. Inspekcja metod — `methods`**

### Пример


In [ ]:
methods(describe_типe)


### Zastosowanie

`methods(f)` pokazuje wszystkie znane методы функции `f`.

To bardzo ważne narzędzie do nauki i debugowania dispatch.


## **9. `@which` — która метод zostanie wywołana?**

### Пример


In [ ]:
@which describe_типe(10)


In [ ]:
@which describe_типe("Julia")


### Анализ

`@which` pozwala bezpośrednio zobaczyć metodę wybraną dla конкретныйch аргументów.


## **10. `applicable`**

### Теория

`applicable(f, args...)` проверяет, czy istnieje метод możliwa do вызовы dla danych аргументów.


In [ ]:
only_ints(x::Integer, y::Integer) = x + y

@show applicable(only_ints, 1, 2)
@show applicable(only_ints, 1.0, 2.0)


## **11. `hasmethod`**

### Теория

`hasmethod` pozwala sprawdzić, czy функция ma metodę zgodną z określonym tuple типów.


In [ ]:
@show hasmethod(only_ints, Tuple{Int, Int})
@show hasmethod(only_ints, Tuple{Float64, Float64})


## **12. Dispatch dla wielu аргументów**

### Пример


In [ ]:
compare_values(x::Integer, y::Integer) = "dwie liczby całkowite"
compare_values(x::Real, y::Real) = "dwie liczby rzeczywiste"
compare_values(x::Real, y::AbstractString) = "liczba i строка"
compare_values(x, y) = "inne połączenie"

@show compare_values(1, 2)
@show compare_values(1.5, 2.5)
@show compare_values(1, "Julia")
@show compare_values([1], :symbol)


### Важно

Wybór nie zależy od „pierwszego аргументu plus reszta”.

Julia analizuje cały zestaw типów аргументów.


## **13. Пример geometryczny**

### Typy


In [ ]:
abstract типe Shape end

struct Circle <: Shape
    radius::Float64
end

struct Rectangle <: Shape
    width::Float64
    height::Float64
end


### Поле powierzchni


In [ ]:
area(s::Circle) = π * s.radius^2
area(s::Rectangle) = s.width * s.height

c = Circle(2.0)
r = Rectangle(3.0, 4.0)

@show area(c)
@show area(r)


### Interakcja dwóch figur


In [ ]:
relation(a::Circle, b::Circle) = "dwa okręgi"
relation(a::Rectangle, b::Rectangle) = "dwa prostokąty"
relation(a::Circle, b::Rectangle) = "okrąg i prostokąt"
relation(a::Rectangle, b::Circle) = "prostokąt i okrąg"

@show relation(c, Circle(1.0))
@show relation(c, r)
@show relation(r, c)


## **14. Parametryczne методы i `where`**

### Пример


In [ ]:
same_типe_add(x::T, y::T) where {T<:Number} = x + y

@show same_типe_add(1, 2)
@show same_типe_add(1.5, 2.5)


### Анализ

`where {T<:Number}` oznacza:

- istnieje pewien тип `T`;
- `T` jest podтипem `Number`;
- oba аргументy mają dokładnie ten sam тип `T`.


## **15. Ograniczanie типów — kiedy ma sens?**

### Хорошая практика

Ograniczenia типów dodajemy wtedy, gdy:

- zachowanie rzeczywiście zależy od типu;
- chcemy zопределять konkretną metodę;
- potrzebujemy jednoznacznego kontraktu.


### Типичная ошибка

Nie pisz:

```julia
f(x::Int) = x + 1
```

jeżeli функция równie dobrze działa dla wszystkich чисел.

Lepsza wersja может być:


In [ ]:
increment_number(x::Number) = x + one(x)

@show increment_number(10)
@show increment_number(3.5)


## **16. Dispatch zależny od значения — przez типy**

### Теория

Dispatch w Julia działa na **типach**, nie bezpośrednio na arbitralnych значенияach.

Jeżeli zachowanie ma zależeć od значения, zwykle stosujemy:

- zwykły `if`;
- albo kodujemy значение w типie.


### Пример z типami znacznikowymi


In [ ]:
abstract типe Operation end
struct AddOperation <: Operation end
struct MultiplyOperation <: Operation end

apply(::AddOperation, x, y) = x + y
apply(::MultiplyOperation, x, y) = x * y

@show apply(AddOperation(), 3, 4)
@show apply(MultiplyOperation(), 3, 4)


## **17. `Val` — значение jako część типu**

### Теория

`Val{x}` pozwala przenieść małą значение do systemu типów.


In [ ]:
operation(::Val{:add}, x, y) = x + y
operation(::Val{:mul}, x, y) = x * y

@show operation(Val(:add), 2, 3)
@show operation(Val(:mul), 2, 3)


### Важно

`Val` jest narzędziem specjalistycznym.

Nie следует zastępować nim zwykłych warunków tylko dlatego, że istnieje.


## **18. Niejednoznaczność metod — ambiguities**

### Теория

Ambiguity powstaje, gdy dla danego вызовы dwie методы są równie specyficzne i Julia nie может jednoznacznie wybrać jednej.


### Пример problemu


In [ ]:
ambiguous_demo(x::Integer, y::Real) = "Integer, Real"
ambiguous_demo(x::Real, y::Integer) = "Real, Integer"


Wywołanie:

```julia
неоднозначный_demo(1, 1)
```

pasuje do obu metod.

Aby uniknąć pozostawiania notebooka w stanie błędu, sprawdzimy to bezpiecznie:


In [ ]:
ambiguity_detected = try
    ambiguous_demo(1, 1)
    false
catch e
    e isa MethodError
end

@show ambiguity_detected


## **19. Jak usuwać ambiguities?**

### Rozwiązanie

Dodaj najbardziej specyficzną metodę:


In [ ]:
ambiguous_demo(x::Integer, y::Integer) = "Integer, Integer"

@show ambiguous_demo(1, 1)


### Хорошая практика

Jeżeli dwa przecięcia типów są logicznie możliwe, zdefiniuj metodę dla ich przecięcia.


## **20. Promotion i dispatch**

### Теория

W obliczeniach numerycznych często mamy różne типy:

```julia
1       # Int
2.5     # Float64
```

Julia posiada mechanizm promocji типów, który pozwala znaleźć wspólny тип reprezentacji.


In [ ]:
@show promote(1, 2.5)
@show promote_типe(Int, Float64)


### Пример z własną funkcją


In [ ]:
function add_promoted(x::Number, y::Number)
    xp, yp = promote(x, y)
    return xp + yp
end

@show add_promoted(1, 2.5)


## **21. `convert` i dispatch**

### Теория

`convert(T, x)` sam jest funkcją korzystającą z metod.

Możemy rozszerzyć mechanizm konwersji dla własnego типu.


In [ ]:
struct Celsius
    value::Float64
end

Base.convert(::Type{Celsius}, x::Real) = Celsius(Float64(x))

celsius = convert(Celsius, 36)

@show celsius
@show celsius.value


### Важно

Rozszerzając funkcje z `Base`, trzeba zachować poprawną semantykę i istniejące konwencje.


## **22. Traits — wzorzec projektowy oparty na dispatch**

### Теория

Trait to sposób opisania właściwości типu, która niekoniecznie wynika bezpośrednio z hierarchii dziedziczenia.


In [ ]:
abstract типe StorageTrait end
struct MutableStorage <: StorageTrait end
struct ImmutableStorage <: StorageTrait end

storage_trait(::Type{<:AbstractVector}) = MutableStorage()
storage_trait(::Type{<:Tuple}) = ImmutableStorage()

storage_message(x) = storage_message(storage_trait(типeof(x)), x)

storage_message(::MutableStorage, x) = "kolekcja mutowalna"
storage_message(::ImmutableStorage, x) = "kolekcja niemutowalna"

@show storage_message([1, 2, 3])
@show storage_message((1, 2, 3))


### Анализ

Najpierw określamy trait, a następnie dispatchujemy na jego типie.

To zaawansowany, ale bardzo użyteczny wzorzec w projektowaniu rozszerzalnych bibliotek.


## **23. Dispatch a `if x isa ...`**

### Podejście warunkowe


In [ ]:
function describe_with_if(x)
    if x isa Integer
        return "integer"
    elseif x isa AbstractFloat
        return "float"
    elseif x isa AbstractString
        return "string"
    else
        return "other"
    end
end


### Podejście przez dispatch


In [ ]:
describe_dispatch(x::Integer) = "integer"
describe_dispatch(x::AbstractFloat) = "float"
describe_dispatch(x::AbstractString) = "string"
describe_dispatch(x) = "other"

@assert describe_with_if(10) == describe_dispatch(10)
@assert describe_with_if(3.14) == describe_dispatch(3.14)
@assert describe_with_if("Julia") == describe_dispatch("Julia")


### Kiedy wybrać które podejście?

- decyzja zależy od **значения** → `if`;
- decyzja zależy od **типu** → często dispatch;
- prosty jednorazowy warunek → `if` может być czytelniejszy;
- rozszerzalne API → dispatch zwykle skaluje się lepiej.


## **24. Dispatch a wydajność**

### Теория

Multiple dispatch nie jest tylko mechanizmem organizacji kodu.

W połączeniu z kompilacją specjalizowaną pozwala Julii generować kod dopasowany do конкретныйch типów аргументów.


### Важно

Nie oznacza to, że „im więcej metod, tym szybciej”.

Wydajność zależy m.in. od:

- stabilności типów;
- przewidywalności dispatch;
- alokacji;
- struktury danych;
- sposobu строкаania kodu.


## **25. Projektowanie rozszerzalnego API**

### Цель

Dobrze zaprojektowana функция powinna umożliwiać dodawanie nowych типów bez modyfikowania istniejącego kodu.


In [ ]:
abstract типe Animal end

struct Dog <: Animal
    name::String
end

struct Cat <: Animal
    name::String
end

speak(a::Dog) = "$(a.name): hau!"
speak(a::Cat) = "$(a.name): miau!"

@show speak(Dog("Rex"))
@show speak(Cat("Luna"))


### Rozszerzenie przez nowy тип

Dodajemy nowy тип i metodę:


In [ ]:
struct Duck <: Animal
    name::String
end

speak(a::Duck) = "$(a.name): kwa!"

@show speak(Duck("Donald"))


### Анализ

Nie musieliśmy zmieniać istniejącej функции warunkowej.

Dodaliśmy nową metodę dla nowego типu.

To jedna z największych zalet multiple dispatch.


## **26. Typowe błędy początkujących**

| Błąd | Przyczyna | Poprawne podejście |
|---|---|---|
| mylenie функции z metodą | jedna функция может mieć wiele implementacji | użyj `methods(f)` |
| zbyt wąskie типy | функция niepotrzebnie ograniczona | użyj типu abstrakcyjnego |
| ogromny `if x isa ...` | zachowanie zależy od типów | rozważ dispatch |
| brak fallback | nieobsłużony тип kończy się `MethodError` | dodaj fallback, jeśli ma sens |
| zbyt szeroki fallback | ukrywa błędy projektu | stosuj świadomie |
| ambiguities | przecinające się сигнатуры | dodaj bardziej specyficzną metodę |
| używanie `Val` do wszystkiego | komplikacja projektu | stosuj tylko tam, gdzie to uzasadnione |
| oczekiwanie dispatch po значения | dispatch działa po типach | użyj `if` lub encode value in типe |


## **27. Практика**

### Задание 27.1 — `describe_number`

Zdefiniuj funkcję z методmi dla:

- `Integer`,
- `AbstractFloat`,
- `Complex`,
- fallback dla innych типów.


In [ ]:
# Ваше решение:


### Примерowe rozwiązanie 27.1


In [ ]:
describe_number(x::Integer) = "integer"
describe_number(x::AbstractFloat) = "float"
describe_number(x::Complex) = "complex"
describe_number(x) = "other"

@assert describe_number(1) == "integer"
@assert describe_number(1.5) == "float"
@assert describe_number(1 + 2im) == "complex"
@assert describe_number("Julia") == "other"


### Задание 27.2 — dwa аргументy

Zdefiniuj `combine_values(x, y)` tak, aby:

- dwa `Integer` były dodawane;
- dwa `String` były konkatenowane;
- pozostałe kombinacje возвращаетły `"unsupported"`.


In [ ]:
# Ваше решение:


### Примерowe rozwiązanie 27.2


In [ ]:
combine_values(x::Integer, y::Integer) = x + y
combine_values(x::AbstractString, y::AbstractString) = x * y
combine_values(x, y) = "unsupported"

@assert combine_values(2, 3) == 5
@assert combine_values("Julia ", "Lang") == "Julia Lang"
@assert combine_values(1, "Julia") == "unsupported"


### Задание 27.3 — `@which`

Sprawdź, która метод `combine_values` zostanie użyta dla:

```julia
combine_values(2, 3)
```


In [ ]:
@which combine_values(2, 3)


### Задание 27.4 — ambiguity

Utwórz dwie przecinające się методы:

```julia
f(x::Integer, y::Real)
f(x::Real, y::Integer)
```

Następnie dodaj trzecią metodę rozwiązującą konflikt dla `(Integer, Integer)`.


In [ ]:
# Ваше решение:


### Примерowe rozwiązanie 27.4


In [ ]:
dispatch_test(x::Integer, y::Real) = "Integer, Real"
dispatch_test(x::Real, y::Integer) = "Real, Integer"
dispatch_test(x::Integer, y::Integer) = "Integer, Integer"

@assert dispatch_test(1, 1) == "Integer, Integer"


### Задание 27.5 — geometria

Dodaj тип:

```julia
struct Square <: Shape
    side::Float64
end
```

oraz metodę `area`.


In [ ]:
# Ваше решение:


### Примерowe rozwiązanie 27.5


In [ ]:
struct Square <: Shape
    side::Float64
end

area(s::Square) = s.side^2

@assert area(Square(5.0)) == 25.0


## **28. Мини-проект — system płatności oparty na dispatch**

### Цель

Zaprojektujemy API, w którym różne методы płatności mają różne zachowanie.


In [ ]:
abstract типe PaymentMethod end

struct CardPayment <: PaymentMethod
    last4::String
end

struct BankTransfer <: PaymentMethod
    iban::String
end

struct CashPayment <: PaymentMethod
end


### Metody `pay`


In [ ]:
pay(method::CardPayment, amount::Real) =
    "Płatność kartą ****$(method.last4): $(round(amount, digits=2)) PLN"

pay(method::BankTransfer, amount::Real) =
    "Przelew na $(method.iban): $(round(amount, digits=2)) PLN"

pay(::CashPayment, amount::Real) =
    "Płatność gotówką: $(round(amount, digits=2)) PLN"


In [ ]:
card = CardPayment("4242")
transfer = BankTransfer("PL001234567890")
cash = CashPayment()

@show pay(card, 199.99)
@show pay(transfer, 500)
@show pay(cash, 50)


### Rozszerzenie systemu

Dodajemy nową metodę płatności bez zmiany istniejących metod:


In [ ]:
struct CryptoPayment <: PaymentMethod
    asset::Symbol
end

pay(method::CryptoPayment, amount::Real) =
    "Płatność $(method.asset): $(round(amount, digits=2)) PLN"

@show pay(CryptoPayment(:BTC), 1000)


### Анализ

Мини-проект pokazuje kluczową własność multiple dispatch:

**system można rozszerzać przez dodawanie nowych типów i metod bez modyfikowania starego kodu.**


## **29. Итоговый checkpoint**

Odpowiedz bez uruchamiania kodu:

1. Czym różni się функция od методы?
2. Co oznacza multiple dispatch?
3. Na ilu аргументach может opierać się wybór методы?
4. Co oznacza „najbardziej specyficzna метод”?
5. Czym różni się тип абстрактный od konkretnego?
6. Po co stosuje się fallback methods?
7. Co pokazuje `methods(f)`?
8. Do czego służy `@which`?
9. Co проверяет `applicable`?
10. Co проверяет `hasmethod`?
11. Co oznacza `where {T<:Number}`?
12. Czy dispatch działa bezpośrednio na значенияach?
13. Do czego służy `Val`?
14. Co to jest ambiguity?
15. Jak usunąć ambiguity?
16. Do czego służy promotion?
17. Jakie отношения ma `convert` z dispatch?
18. Czym jest trait?
19. Kiedy lepszy jest `if`, a kiedy dispatch?
20. Dlaczego multiple dispatch dobrze wspiera rozszerzalne API?


## **30. Итоги lekcji**

Najważniejsze zasady Lesson 8:

1. Funkcja может mieć wiele metod.
2. Julia выбирает metodę na podstawie типów wszystkich аргументów.
3. Najbardziej specyficzna pasująca метод ma pierwszeństwo.
4. Typy abstrakcyjne umożliwiają definiowanie metod dla całych rodzin типów.
5. Fallback methods pomagają obsługiwać przypadki ogólne.
6. `methods`, `@which`, `applicable` i `hasmethod` są kluczowymi narzędziami inspekcji.
7. `where` pozwala определять зависимости między типami аргументów.
8. Dispatch działa po типach, nie po arbitralnych значенияach.
9. `Val` может przenieść małą значение do systemu типów, ale nie следует go nadиспользовать.
10. Ambiguities powstają przy przecinających się, równie specyficznych методch.
11. Promotion pomaga ujednolicić типy w operacjach numerycznych.
12. `convert` również opiera się na mechanizmie metod.
13. Traits pozwalają modelować właściwości niezależne od prostej hierarchii типów.
14. Jeżeli zachowanie zależy od типu, dispatch często jest lepszy niż duży `if x isa`.
15. Multiple dispatch jest jednym z najważniejszych mechanizmów projektowania rozszerzalnych bibliotek Julia.


## Источники для дальнейшего изучения

- Julia Manual — Methods
- Julia Manual — Types
- Julia Manual — Conversion and Promotion
- Julia Manual — Interfaces
- Julia Base — `methods`
- Julia InteractiveUtils — `@which`
- Julia Base — `applicable`
- Julia Base — `hasmethod`


---

**Cartesian School · Julia Course**  
**Lesson 8 — Multiple Dispatch**  
**Автор:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School

[← Lesson 7 — Plotting](Lesson_7_Plotting_Julia_Cartesian_School_RU.ipynb)  
[Оглавление](../README.ru.md)  
[Lesson 9 — Julia is Fast →](Lesson_9_Julia_is_Fast_Cartesian_School_RU.ipynb)
